## Section 2b: Task 3 — Item Recommendation

### Why Content-Based Filtering?

There are three common ways to build a recommendation system. Here is a plain-language
comparison and why we chose content-based filtering for this project:

| Approach | How it works | Why not chosen here |
|---|---|---|
| **Collaborative Filtering** | "People who liked X also liked Y" — finds patterns across many users' histories | Our dataset has no persistent user identity; we cannot build cross-user purchase patterns |
| **Knowledge-Based** | Recommends by fixed rules: same category, same price range, same brand | Too rigid — two products can be very similar in spirit (e.g. two hydrating serums from different brands) yet share no explicit attributes |
| **Content-Based Filtering ✓** | Compares products by *what reviewers say about them* | Works directly with our 61 K review texts; no user history needed; naturally captures nuanced attributes like texture, finish, and skin type |

**Why GloVe vectors instead of raw word counts (Bag-of-Words)?**

Bag-of-Words treats every word as independent — "moisturising" and "hydrating" would be
completely different features even though they mean nearly the same thing.
GloVe vectors encode semantic meaning: words with similar meaning end up close together
in the 300-dimensional space. This means products described with different but related
vocabulary are still correctly identified as similar.

### How It Works — Plain Language

When a shopper selects a product, the system compares it against all 294 other products
in the catalogue using three signals: **what reviewers say about it** (70 %),
**whether it is the same brand** (15 %), and **whether it is similarly priced** (15 %).

Each product is represented by a single numerical "fingerprint" — a 300-number vector
computed from the average meaning of all words across all its reviews.
The system measures how similar two fingerprints are (cosine similarity), combines
that with the brand and price signals, and ranks every other product from most to
least similar.  The top results are shown as recommendations.

No user data or purchase history is needed — the recommendations are driven entirely
by product content.

### Hybrid Similarity Formula

$$\text{score} = 0.70 \times \text{text\_cosine} + 0.15 \times \text{brand\_match} + 0.15 \times \text{price\_sim}$$

| Component | Weight | How it is computed |
|---|---|---|
| `text_cosine` | **70 %** | Cosine similarity between two 300-dim product GloVe vectors — ranges 0 (unrelated) to 1 (identical meaning) |
| `brand_match` | **15 %** | Binary: 1.0 if same brand, 0.0 otherwise |
| `price_sim` | **15 %** | `max(0, 1 − |p₁−p₂| / max(p₁,p₂))` — 1.0 for identical price, 0.0 when one product costs double the other |

**Why this weighting?**
Review language is by far the richest signal — it captures texture, finish, skin type
compatibility, and user experience in a way that brand and price cannot.
Brand and price act as tie-breakers: if two products have very similar review language,
a shopper would naturally prefer one from the same brand or in the same price range.
Equal 15 % weight for each reflects their supporting (not primary) role.

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


def get_recommendations(product_id, product_vectors, products_df, n=5):
    """
    Return the top-N most similar products to `product_id`.

    Args:
        product_id:      ID of the selected product
        product_vectors: dict[product_id → np.array(300)]  (from product_vectors.pkl)
        products_df:     DataFrame with product metadata (brand_name, price, …)
        n:               number of recommendations to return

    Returns:
        list of dicts: [{product_id, brand_name, product_title, price,
                         avg_product_rating, review_count, similarity_score}, ...]
    """
    if product_id not in product_vectors:
        return []

    target_row = products_df[products_df['product_id'] == product_id]
    if target_row.empty:
        return []

    target_vector = product_vectors[product_id].reshape(1, -1)
    target_brand  = target_row.iloc[0]['brand_name']
    target_price  = float(target_row.iloc[0]['price'])

    # Vectorised cosine similarity against all other products at once
    other_ids    = [pid for pid in product_vectors if pid != product_id]
    other_matrix = np.array([product_vectors[pid] for pid in other_ids])
    text_sims    = cosine_similarity(target_vector, other_matrix)[0]   # shape (N-1,)

    prod_lookup  = products_df.set_index('product_id')
    candidates   = []

    for i, other_id in enumerate(other_ids):
        if other_id not in prod_lookup.index:
            continue

        other_row   = prod_lookup.loc[other_id]
        other_price = float(other_row['price'])

        # Brand match (binary)
        brand_match = 1.0 if target_brand == other_row['brand_name'] else 0.0

        # Price similarity — normalised by the larger price
        if target_price == 0 and other_price == 0:
            price_sim = 1.0
        elif target_price == 0 or other_price == 0:
            price_sim = 0.0
        else:
            price_sim = max(0.0, 1.0 - abs(target_price - other_price) / max(target_price, other_price))

        # Hybrid score
        final_score = 0.70 * float(text_sims[i]) + 0.15 * brand_match + 0.15 * price_sim

        candidates.append({
            'product_id':         other_id,
            'brand_name':         other_row['brand_name'],
            'product_title':      other_row['product_title'],
            'price':              other_row['price'],
            'avg_product_rating': other_row['avg_product_rating'],
            'review_count':       other_row['review_count'],
            'similarity_score':   final_score,
        })

    candidates.sort(key=lambda x: x['similarity_score'], reverse=True)
    return candidates[:n]

### Runtime Pipeline

The recommendation engine requires **no training at runtime**.
`product_vectors.pkl` — built in Section 1 Step 9 — is the only dependency.
Here is the full data flow from training to serving a recommendation:

```
OFFLINE  (scripts/train_models.py)
  processed reviews  →  mean GloVe per review  →  mean per product  →  product_vectors.pkl

APP STARTUP  (core/data_loader.load_all())
  product_vectors.pkl  →  load into memory  →  inject into Gradio tab

AT REQUEST TIME  (get_recommendations)
  user selects product  →  cosine similarity vs 294 others
                        →  + brand match + price similarity
                        →  sort by hybrid score  →  top-N results
```

For a 295-product catalogue this entire computation takes under 1 ms at request time.

In [ ]:
# Assumes the app has already called data = load_all()
product_vectors = data['models']['product_vectors']   # dict[product_id → np.array(300)]
products_df     = data['products_df']                  # 295-row DataFrame

# Get top-5 similar products for an example product
example_pid     = products_df.iloc[0]['product_id']
recommendations = get_recommendations(
    product_id      = example_pid,
    product_vectors = product_vectors,
    products_df     = products_df,
    n               = 5
)

for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec['product_title']}")
    print(f"   Brand: {rec['brand_name']}  |  Price: ₹{rec['price']}"
          f"  |  Similarity: {rec['similarity_score']:.3f}\n")